# ATHENA Data Engineering Pipeline

This notebook builds ATHENA's **Cyber Threat Actor Master Dataset** combining CTI information about cyber threat actors' behavior from 4 CTI data souces and geopolitical context indicators from 1 global event data set. Scores are calculated for each actor to allow cyber defenders to rank and priortize cyber actors by threat level. The resulting dataset is the ATHENA model for this praxis.

Each actor row is enriched with:

| Data Source | What it adds |
|---|---|
| MITRE ATT&CK | Actor names, aliases, techniques, tactics, sector targets, country attribution, actor type |
| AlienVault OTX | Candidate CVE associations that are then manually reviewed |
| NVD bulk feed | CVSS severity scores for each CVE |
| CISA KEV | Identifies CVEs with confirmed real-world exploitation |
| ACLED | Country-level political violence data → geopolitical instability scores |

Data files are stored in Google MyDrive along with this notebook:

- **Input folder:** `/content/drive/MyDrive/Colab Notebooks/input`  
- **Output folder:** `/content/drive/MyDrive/Colab Notebooks/output`

**Input files:**
- `acled.xlsx` — ACLED monthly summary (columns: COUNTRY, MONTH, YEAR, EVENTS)
- `known_exploited_vulnerabilities.csv` — CISA KEV catalog export
- One or more NVD JSON or ZIP feed files (e.g. `nvdcve-2.0-recent.json.zip`)
- *(Optional)* `actor_cve_mapping.csv` — the reviewed actor-to-CVE links if the code has already run before

**Final output:** `actor_master_dataset_final.csv`
- Additional files are created during the process of this code and stored in the output folder as well (i.e. actor_sector_mapping.csv, actor_alias_mapping.csv, etc.)

**Note:** Code is broken out into different cells and commented along the way to explain the process, interim files created, etc. as much as possible. The searching Alienvault OTX code (cell 12) can be skipped if already done or to save time. Depending on how you set the OTX search settings in cell 3 it can take a long time (8-10 hours) due to the API calls, inherent noise of CTI reports, and searching multiple aliases for all actor groups. Code is explained in comments and text cells throughout.

## Cell 1: Mount Google Drive

Connects this notebook to Google Drive so it can read input files
and write outputs to the folders defined in Cell 3.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


## Cell 2: Install & Import Libraries

This cell installs `pycountry` (used for ISO country name matching) and imports  Python packages used throughout the notebook.


In [ ]:
# pycountry provides a database of ISO country names and codes used for country_keyword
try:
    import pycountry
except ImportError:
    !pip install pycountry -q
    import pycountry

import os
import re
import json
import glob
import time
import zipfile

import requests
import pandas as pd
import numpy as np
from urllib.parse import quote

# userdata lets us read secrets stored in Colab's secret manager
from google.colab import userdata

print("All libraries imported.")


All libraries imported.


## Cell 3: Configuration & Settings

This cells sets up the file paths for the input and output directories, file names for the input files, and allows for changing the settings for the OTX search.

| OTX Setting | Purpose |
|---|---|
| `RUN_OTX` | `True` = run the OTX actor search; `False` = skip it |
| `MAX_ACTORS_TO_SEARCH` | How many actors to query OTX for (`None` = all actors) |
| `MAX_ALIASES_PER_ACTOR` | Max alternate names searched per actor |
| `MAX_PAGES_PER_QUERY` | OTX result pages fetched per search term |
| `REQUEST_DELAY_SECONDS` | Pause between OTX requests (avoids rate-limit errors) |


In [ ]:
# Configurations and Settings
import os
import re
import json
import glob
import time
import zipfile

import requests
import pandas as pd
import numpy as np
from urllib.parse import quote

from google.colab import userdata

# ---Folder layout---
BASE_DIR   = "/content/drive/MyDrive/Colab Notebooks/"
INPUT_DIR  = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

os.makedirs(INPUT_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input  folder:", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)

# ---Input file paths---
# ACLED monthly political violence summary
# Data from https://acleddata.com/aggregated/number-political-violence-events-country-year
# Expected columns: COUNTRY, MONTH, YEAR, EVENTS
ACLED_FILE             = os.path.join(INPUT_DIR, "ACLED.xlsx")

# CISA Known Exploited Vulnerabilities export
# Data from https://www.cisa.gov/known-exploited-vulnerabilities-catalog
CISA_KEV_FILE          = os.path.join(INPUT_DIR, "known_exploited_vulnerabilities.csv")

# The reviewed actor-to-CVE mapping (produced from OTX candidates in a prior run)
# This file is OPTIONAL — missing it sets all vulnerability features to zero.
ACTOR_CVE_MAPPING_FILE = os.path.join(INPUT_DIR, "actor_cve_mapping.csv")

# MITRE ATT&CK data
# The notebook downloads this automatically from MITREs GITHUB
ATTACK_ENTERPRISE_URL = (
    "https://raw.githubusercontent.com/mitre-attack/attack-stix-data"
    "/master/enterprise-attack/enterprise-attack.json"
)

# ---OTX search settings---
RUN_OTX               = False   # Set to False to skip the OTX step entirely
MAX_ACTORS_TO_SEARCH  = None     # Set to None to search all actors (takes a long time)
MAX_ALIASES_PER_ACTOR = 6      # How many alternate names to try per actor
MAX_PAGES_PER_QUERY   = 2      # OTX returns paginated results; this caps pages fetched
REQUEST_DELAY_SECONDS = 5.0    # Seconds to wait between OTX API calls


Input  folder: /content/drive/MyDrive/Colab Notebooks/input
Output folder: /content/drive/MyDrive/Colab Notebooks/output


## Cell 4: Helper Functions

Reusable functions used throughout the pipeline


In [ ]:
# Output path helper
def out_path(filename):
    """Return the full path for a file in the output folder."""
    return os.path.join(OUTPUT_DIR, filename)

# Input file checker
def input_exists(path):
    """Print whether an input file is present and return True/False."""
    exists = os.path.exists(path)
    status = " FOUND  " if exists else " MISSING"
    print(f"  {status}: {path}")
    return exists

# MITRE ATT&CK STIX helpers
def get_attack_id(stix_object):
    """
    Extract the ATT&CK ID (e.g. G0016, T1059) from a STIX object.
    ATT&CK IDs live inside the object's external_references list.
    """
    for ref in stix_object.get("external_references", []):
        if ref.get("source_name") == "mitre-attack" and ref.get("external_id"):
            return ref["external_id"]
    return None

def get_attack_url(stix_object):
    """Return the ATT&CK website URL from a STIX object's external references."""
    for ref in stix_object.get("external_references", []):
        if ref.get("source_name") == "mitre-attack" and ref.get("url"):
            return ref["url"]
    return ""

# Data-cleaning helpers
def clean_date(value):
    """
    Convert any date-like value to a YYYY-MM-DD string.
    Returns an empty string for missing values.
    """
    if value is None or pd.isna(value):
        return ""
    return str(value)[:10]

def split_list(value):
    """
    Split a semicolon-, comma-, or pipe-delimited string into a Python list.
    Works correctly on actual Python lists too (passed through unchanged).
    """
    if value is None or pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    return [x.strip() for x in re.split(r"[;,|]", str(value)) if x.strip()]

def normalize_series(series):
    """
    Scale a numeric column to a 0–1 range (min-max normalization).
    Used when combining features with different units into a single score.
    Returns all zeros if every row has the same value (avoids divide-by-zero).
    """
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.max() == series.min():
        return pd.Series([0.0] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min())

# Text-matching helpers
def text_contains_any(text, keywords):
    """Return True if any keyword appears (case-insensitive) anywhere in text."""
    text = str(text).lower()
    return any(kw.lower() in text for kw in keywords)

def contains_phrase(text, phrase):
    """
    Return True if phrase appears as a whole word in text (case-insensitive).
    Uses a word-boundary regex so 'Iran' doesn't match 'Ukraine'.
    """
    return bool(
        re.search(r"\b" + re.escape(str(phrase).lower()) + r"\b",
                  str(text).lower())
    )

print("Helper functions defined")


Helper functions defined


## Cell 5: Verify Input Files

This cell checks that each expected input file exists in the Google Drive input folder. Missing files will not stop the pipeline and will contribute empty/zero features to the final dataset.

This cell also looks for NVD related files and de-duplicates.
* NVD data was pulled from the NIST NVD website at https://nvd.nist.gov/vuln/data-feeds
* The bulk download of the full NVD database is accessed via API later in Cell 13.
* But, NVD data files can be included in the Input Data folder on Google Drive along with ACLED and CISA KEV mentioned in Cell 3. This is helpful if you want to include recent updates or modifications to the NVD that NIST provides regularly in separate files.



In [ ]:
print("Checking input files...\n")
input_exists(ACLED_FILE)
input_exists(CISA_KEV_FILE)
input_exists(ACTOR_CVE_MAPPING_FILE)   # file is optional

# Scan for NVD JSON / ZIP feed files using common naming patterns
print("\nLooking for NVD feed files in input folder...")
nvd_candidate_files = []
for pattern in ["*nvd*.json", "*nvd*.zip", "*nvdcve*.json",
                 "*nvdcve*.zip", "*CVE-*.json", "*CVE-*.zip",
                 "*cve*.json", "*cve*.zip"]:
    nvd_candidate_files.extend(glob.glob(os.path.join(INPUT_DIR, pattern)))

# Remove duplicates while preserving order
nvd_candidate_files = list(dict.fromkeys(nvd_candidate_files))

if nvd_candidate_files:
    for f in nvd_candidate_files:
        print(f"   FOUND NVD: {f}")
else:
    print("  No NVD JSON/ZIP files found — CVSS scores will be unavailable.")


Checking input files...

   FOUND  : /content/drive/MyDrive/Colab Notebooks/input/ACLED.xlsx
   FOUND  : /content/drive/MyDrive/Colab Notebooks/input/known_exploited_vulnerabilities.csv
   FOUND  : /content/drive/MyDrive/Colab Notebooks/input/actor_cve_mapping.csv

Looking for NVD feed files in input folder...
   FOUND NVD: /content/drive/MyDrive/Colab Notebooks/input/nvdcve-2.0-modified.json
   FOUND NVD: /content/drive/MyDrive/Colab Notebooks/input/nvd_full.json


## Cell 6: Country & Sector Keyword Definitions

This cell builds the reference lists used to extract **country attribution**, **targeted sectors**, and **actor type** from MITRE ATT&CK group description text.

How it works:
- Country names come from the ISO 3166 standard (via `pycountry`) plus an addiitonal alias list (e.g. "DPRK" → North Korea)
- Sector names follow CISA's 16 critical infrastructure sectors, extended with
  Education, Media, and NGO
- Actor type includes three keyword types (State, Nonstate Cybercriminal, and Nonstate Havctivist) and is inferred from phrases like  "state-sponsored" or "financially motivated" used to describe actors in CTI reporting.

The dictionaries can be expanded to include more keywords as needed.

In [ ]:
# ---Country Keywords---
# Build ISO country name list
# pycountry gives us official, common, and short names for every UN member state.
# Extra entries for names and adjective forms common in CTI writing are also included
def build_iso_country_list():
    countries = set()
    for c in pycountry.countries:
        countries.add(c.name)
        if hasattr(c, "official_name"): countries.add(c.official_name)
        if hasattr(c, "common_name"):   countries.add(c.common_name)

    countries.update([
        "Russia", "Russian Federation",
        "China", "People's Republic of China", "PRC",
        "Iran", "Islamic Republic of Iran",
        "North Korea", "DPRK", "Democratic People's Republic of Korea",
        "South Korea", "Republic of Korea",
        "United States", "United States of America", "USA", "U.S.",
        "United Kingdom", "UK", "Great Britain",
        "Vietnam", "Viet Nam",
        "Palestine", "Gaza",
    ])
    # Longest names first so multi-word matches take priority
    return sorted(countries, key=len, reverse=True)

ALL_COUNTRIES = build_iso_country_list()

# Maps adjective/alias forms → canonical country name
# Add new aliases here to improve country detection
COUNTRY_ALIASES = {
    "russian": "Russia",            "russian federation": "Russia",
    "chinese": "China",             "people's republic of china": "China",
    "prc": "China",
    "iranian": "Iran",
    "north korean": "North Korea",  "dprk": "North Korea",
    "democratic people's republic of korea": "North Korea",
    "south korean": "South Korea",  "republic of korea": "South Korea",
    "american": "United States",    "u.s.": "United States",
    "usa": "United States",
    "british": "United Kingdom",    "uk": "United Kingdom",
    "great britain": "United Kingdom",
    "vietnamese": "Vietnam",
    "turkish": "Turkey",
    "pakistani": "Pakistan",
    "indian": "India",
    "israeli": "Israel",
    "palestinian": "Palestine",     "gaza": "Palestine",
}

def derive_country_from_mitre_text(description):
    """
    Scan a MITRE group description for country names and aliases.
    Returns a semicolon-joined string of canonical country names, or 'unknown'.
    """
    if description is None or pd.isna(description):
        return "Unknown"
    text = str(description).lower()
    found = set()

    # Match full country names (longest first to avoid partial matches)
    for country in ALL_COUNTRIES:
        if re.search(r"\b" + re.escape(country.lower()) + r"\b", text):
            found.add(COUNTRY_ALIASES.get(country.lower(), country))

    # Match adjective / alias forms
    for alias, canonical in COUNTRY_ALIASES.items():
        if re.search(r"\b" + re.escape(alias) + r"\b", text):
            found.add(canonical)

    return "; ".join(sorted(found)) if found else "Unknown"

# ---Sector Keywords---
# Keywords derived from CISA critical infrastructure sector names
# Values = keywords that indicate an actor targets that sector
# Keywords are matched case-insensitively using substring search
SECTOR_KEYWORDS = {
    "Chemical":
        ["chemical", "chemicals", "petrochemical"],
    "Commercial Facilities":
        ["commercial facilities", "hotel", "casino", "stadium", "retail"],
    "Communications":
        ["communications", "telecommunications", "telecom", "satellite",
         "internet service provider", "isp", "mobile carrier", "media company"],
    "Critical Manufacturing":
        ["critical manufacturing", "manufacturing", "industrial",
         "automotive", "semiconductor", "electronics"],
    "Dams":
        ["dam", "dams", "reservoir"],
    "Defense Industrial Base":
        ["defense industrial base", "defense", "defence", "military",
         "aerospace", "weapons", "defense contractor"],
    "Emergency Services":
        ["emergency services", "police", "fire department", "ems",
         "public safety", "first responder"],
    "Energy":
        ["energy", "oil", "gas", "electric", "electricity", "power",
         "utility", "utilities", "pipeline", "renewable", "nuclear power"],
    "Financial Services":
        ["financial", "finance", "bank", "banking", "insurance",
         "cryptocurrency", "crypto exchange", "payment", "fintech"],
    "Food and Agriculture":
        ["food", "agriculture", "agricultural", "farm", "livestock"],
    "Government Facilities":
        ["government", "public sector", "federal", "ministry",
         "embassy", "diplomatic", "municipal"],
    "Healthcare and Public Health":
        ["healthcare", "health care", "public health", "hospital",
         "medical", "pharmaceutical", "pharma", "biotech"],
    "Information Technology":
        ["information technology", "technology", "software", "hardware",
         "cloud", "managed service provider", "msp", "saas",
         "cybersecurity company"],
    "Nuclear Reactors, Materials, and Waste":
        ["nuclear reactor", "nuclear materials", "nuclear waste",
         "nuclear facility"],
    "Transportation Systems":
        ["transportation", "aviation", "airline", "airport", "shipping",
         "maritime", "rail", "logistics", "port", "trucking"],
    "Water and Wastewater Systems":
        ["water", "wastewater", "sewage", "water treatment"],
    "Education":
        ["education", "university", "academic", "school",
         "research institution"],
    "Media":
        ["media", "journalist", "news", "press"],
    "NGO":
        ["ngo", "non-governmental", "human rights", "civil society",
         "activist", "think tank"],
}

#---Actor Type Kewords---
# Common words to describe state and nonstate actors found in CTI reporting
# Keywords that suggest the actor is state-sponsored
STATE_KEYWORDS = [
    "state-sponsored", "state sponsored", "nation-state", "nation state",
    "government-backed", "government backed", "attributed to", "sponsored by",
    "associated with the government", "linked to the government",
]

# Keywords that suggest the actor is a non-state criminal group
NONSTATE_CRIME_KEYWORDS = [
    "financially motivated", "criminal", "cybercriminal", "crimeware",
    "ransomware", "hacktivist", "extortion", "ransomware", "cryptojacking", "RaaS"
]

# Keywords that suggest the actor is a non-state hactivist group
NONSTATE_HACK_KEYWORDS = [
    "activist", "hactivism", "hacktivist", "activism"
]

def derive_sectors_from_mitre_text(description):
    """Return semicolon-joined list of matched sector names, or 'unknown'."""
    if description is None or pd.isna(description):
        return "unknown"
    found = [sector for sector, kws in SECTOR_KEYWORDS.items()
             if text_contains_any(description, kws)]
    return "; ".join(sorted(set(found))) if found else "unknown"

def derive_actor_type_from_mitre_text(description):
    """Return actor type based on description keywords."""
    if description is None or pd.isna(description):
        return "Unknown"
    if text_contains_any(description, STATE_KEYWORDS):    return "State"
    if text_contains_any(description, NONSTATE_CRIME_KEYWORDS): return "Nonstate CyberCrime"
    if text_contains_any(description, NONSTATE_HACK_KEYWORDS): return "Nonstate Hactivist"


print(f"Country reference list built: {len(ALL_COUNTRIES):,} entries")
print(f"Sector categories defined:    {len(SECTOR_KEYWORDS)}")


Country reference list built: 435 entries
Sector categories defined:    19


## Cell 7: Download & Parse MITRE ATT&CK Data Set

This cell downloads the full MITRE ATT&CK Enterprise dataset in STIX 2.x JSON format and separates it into three buckets:

- **Groups** (`intrusion-set`) — the threat actor groups; these become the rows
  in the actor master dataset
- **Techniques** (`attack-pattern`) — the individual ATT&CK techniques
- **Relationships** — "group X uses technique Y" links

Revoked and deprecated entries are skipped automatically


In [ ]:
print("Downloading MITRE ATT&CK Enterprise STIX data...")
print("(Source:", ATTACK_ENTERPRISE_URL, ")")

response = requests.get(ATTACK_ENTERPRISE_URL, timeout=60)
response.raise_for_status()
attack_data = response.json()
objects = attack_data.get("objects", [])

print(f"\nTotal STIX objects downloaded: {len(objects):,}")

# Sort into the three types we need
groups_by_stix_id     = {}   # STIX ID → group object    (the primary actors)
techniques_by_stix_id = {}   # STIX ID → technique object
relationships         = []   # all relationship objects

for obj in objects:
    obj_type = obj.get("type")

    if obj_type == "intrusion-set":
        # Skip retired / superseded groups
        if obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        if get_attack_id(obj):
            groups_by_stix_id[obj["id"]] = obj

    elif obj_type == "attack-pattern":
        if obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        if get_attack_id(obj):
            techniques_by_stix_id[obj["id"]] = obj

    elif obj_type == "relationship":
        relationships.append(obj)

print(f"\nActive threat actor groups:  {len(groups_by_stix_id):,}")
print(f"Active ATT&CK techniques:    {len(techniques_by_stix_id):,}")
print(f"Relationship links:          {len(relationships):,}")


(Source: https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json )

Total STIX objects downloaded: 25,843

Active threat actor groups:  174
Active ATT&CK techniques:    697
Relationship links:          21,025


## Cell 8: Build Actor to Technique Mapping
This cell builds the actor to technique mapping from the MITRE ATT&CK data set.
From the ATT&CK relationship objects, this code produces a flat table of
which actor uses which technique, and under which tactics.

From this table the pipeline calculates two behaviour features per actor:
- `technique_count` — number of distinct techniques documented for the actor
- `tactic_count` — number of distinct kill-chain phases (tactics) covered



In [ ]:
technique_rows = []

for rel in relationships:
    # For relationships, the code focuses on "actor uses technique" relationships
    if rel.get("relationship_type") != "uses":
        continue

    source_ref = rel.get("source_ref")   # the actor
    target_ref = rel.get("target_ref")   # the technique

    # Skip if either end isn't in our active sets
    if source_ref not in groups_by_stix_id:     continue
    if target_ref not in techniques_by_stix_id: continue

    group_obj     = groups_by_stix_id[source_ref]
    technique_obj = techniques_by_stix_id[target_ref]

    # Collect the ATT&CK kill-chain phases (tactics) for this technique
    tactics = [
        phase["phase_name"]
        for phase in technique_obj.get("kill_chain_phases", [])
        if phase.get("kill_chain_name") == "mitre-attack"
    ]

    technique_rows.append({
        "group_id"                : get_attack_id(group_obj),
        "actor_name"              : group_obj.get("name", ""),
        "technique_id"            : get_attack_id(technique_obj),
        "technique_name"          : technique_obj.get("name", ""),
        "tactics"                 : "; ".join(sorted(set(tactics))),
        "relationship_description": rel.get("description", ""),
    })

actor_technique_mapping = pd.DataFrame(technique_rows).drop_duplicates()
print(f"Actor-technique rows built: {len(actor_technique_mapping):,}")

# ── Summarise per actor ────────────────────────────────────────────────────────
if len(actor_technique_mapping) > 0:

    # Count distinct techniques per actor
    technique_features = (
        actor_technique_mapping
        .groupby("group_id")
        .agg(technique_count=("technique_id", "nunique"))
        .reset_index()
    )

    # Explode tactic lists so we can count distinct tactics per actor
    tactic_rows = []
    for _, row in actor_technique_mapping.iterrows():
        for tactic in split_list(row["tactics"]):
            tactic_rows.append({"group_id": row["group_id"], "tactic": tactic})

    tactic_df = pd.DataFrame(tactic_rows)

    if len(tactic_df) > 0:
        tactic_features = (
            tactic_df.groupby("group_id")
            .agg(tactic_count=("tactic", "nunique"))
            .reset_index()
        )
    else:
        tactic_features = pd.DataFrame(columns=["group_id", "tactic_count"])

    technique_features = technique_features.merge(tactic_features, on="group_id", how="left")

else:
    technique_features = pd.DataFrame(columns=["group_id", "technique_count", "tactic_count"])

print(f"Actors with technique/tactic data: {len(technique_features):,}")


Actor-technique rows built: 4,546
Actors with technique/tactic data: 170


## Cell 9: Build the Actor Master Dataset from the ATT&CK data
This cell uses the MITRE ATT&CK data to create one row per actor represeted by MITRE ATT&CK group ID. For each actor the pipeline:

1. Pulls structured metadata (name, aliases, first/last seen) directly from the STIX MITRE ATT&CK data.
2. Runs keyword extraction on the free-text description to infer:
   - **Country attribution** (e.g. Russia, China)
   - **Targeted sectors** (e.g. Energy; Government Facilities)
   - **Actor type** (state / nonstate / unknown)
3. Merges in the technique and tactic counts from Cell 8.
4. Computes a `technique_diversity_score` (normalized technique count) as an
   early input to the Capability Score.

5. Extracts CVEs from MITRE ATT&CK technique external references. Some ATT&CK techniques list specific CVEs in their external references. Because actors are already mapped to techniques in Cell 8, the pipeline can follow that link: actor → technique → CVE to derive MITRE-sourced actor-CVE associations. These are saved to actor_cve_mitre.csv and fed into the vulnerability scoring pipeline in Cell 15, combined with user reviewed OTX candidates. MITRE CVEs do not require manual review because they are curated by the ATT&CK team.

Coverage note: The MITRE data is selective. It only links CVEs that are central to a technique definition, so coverage is sparse but high-confidence. OTX candidates (Cell 12) helps fill the gap and merge with another kind of CTI data.

The output is saved as `actor_master_dataset_mitre_stage.csv`


In [ ]:
actor_rows = []

for stix_id, obj in groups_by_stix_id.items():
    group_id    = get_attack_id(obj)
    description = obj.get("description", "")
    aliases     = obj.get("aliases", [])

    actor_rows.append({
        "group_id"                         : group_id,
        "actor_name"                       : obj.get("name", ""),
        "aliases"                          : "; ".join(aliases),
        "actor_type"                       : derive_actor_type_from_mitre_text(description),
        "country_attribution"              : derive_country_from_mitre_text(description),
        "targeted_sectors"                 : derive_sectors_from_mitre_text(description),
        # "first_seen"                       : clean_date(obj.get("first_seen", "")),
        # "last_seen"                        : clean_date(obj.get("last_seen", "")),
        #"mitre_attack_url"                 : get_attack_url(obj),
        "mitre_description"                : description,
    })

actor_master_dataset = (
    pd.DataFrame(actor_rows)
    .sort_values("group_id")
    .reset_index(drop=True)
)

# Merge in technique and tactic counts from Cell 8
actor_master_dataset = actor_master_dataset.merge(
    technique_features, on="group_id", how="left"
)
actor_master_dataset["technique_count"] = (
    actor_master_dataset["technique_count"].fillna(0).astype(int)
)
actor_master_dataset["tactic_count"] = (
    actor_master_dataset["tactic_count"].fillna(0).astype(int)
)

# Normalized technique count — one component of the Capability Score in Cell 19
# technique_diversity_score is how broad an actor's documented attack toolkit is/how many distinct ATT&CK techniques they have been observed using
# Ituses the technique_count value from Cell 8 and is min-max normalized to 0-1 across all actors.
# The actor with the most techniques is 1.0, fewest is 0.0 and other actors scale between them.

actor_master_dataset["technique_diversity_score"] = (
    normalize_series(actor_master_dataset["technique_count"])
)

print(f"Actor master dataset rows: {len(actor_master_dataset):,}")
print(f"Columns: {list(actor_master_dataset.columns)}")

# ---Extract CVEs from MITRE ATT&Ck data to map to actors---
#  CVEs are found in three areas in MITRE data
#   SOURCE A — technique external_references (ALL fields, not just external_id)
#     Some techniques list CVEs as source_name="cve", but more commonly
#     they appear as NVD URLs (e.g. https://nvd.nist.gov/vuln/detail/CVE-2019-11510)
#     or embedded in description text within the ref object.
#
#   SOURCE B — technique description text
#     CVEs as plain text in their descriptions (e.g. "Adversaries exploited CVE-2021-44228 to...")
#
#   SOURCE C — relationship description text
#     When ATT&CK records that group X uses technique Y, the relationship  object sometimes names the specific CVE the group exploited
#     This directly links a named actor to a named CVE and is the best source for the mapping.
#
# All three are scanned below and tagged with their source for transparency.

CVE_RE = re.compile(r"CVE-\d{4}-\d{4,7}", re.IGNORECASE)

def extract_cves_from_string(text):
    """Extract and deduplicate CVE IDs from any string."""
    if not text:
        return []
    return sorted(set(c.upper() for c in CVE_RE.findall(str(text))))

# Build a fast reverse lookup: ATT&CK technique ID → STIX object
technique_id_to_stix = {
    get_attack_id(obj): obj
    for obj in techniques_by_stix_id.values()
}

# Build a fast lookup for relationship descriptions:
# (group_stix_id, technique_stix_id) → description text
rel_desc_lookup = {}
for rel in relationships:
    if rel.get("relationship_type") != "uses":
        continue
    key = (rel.get("source_ref",""), rel.get("target_ref",""))
    desc = rel.get("description","") or ""
    if desc:
        rel_desc_lookup[key] = desc

mitre_cve_rows = []

for _, row in actor_technique_mapping.iterrows():
    technique_obj = technique_id_to_stix.get(row["technique_id"])
    if technique_obj is None:
        continue

    technique_stix_id = technique_obj["id"]
    group_stix_id = None

    # Find the STIX ID for this group (needed to look up relationship descriptions)
    for sid, gobj in groups_by_stix_id.items():
        if get_attack_id(gobj) == row["group_id"]:
            group_stix_id = sid
            break

    # Source A: scan all fields of every external_reference
    # Catches CVEs in source_name="cve", in NVD URLs, and in ref description text
    for ref in technique_obj.get("external_references", []):
        for cve_id in extract_cves_from_string(json.dumps(ref)):
            mitre_cve_rows.append({
                "group_id"      : row["group_id"],
                "actor_name"    : row["actor_name"],
                "cve_id"        : cve_id,
                "technique_id"  : row["technique_id"],
                "technique_name": row["technique_name"],
                "source"        : "MITRE ATT&CK: technique external_reference",
            })

    # Source B: scan technique description text
    for cve_id in extract_cves_from_string(technique_obj.get("description","")):
        mitre_cve_rows.append({
            "group_id"      : row["group_id"],
            "actor_name"    : row["actor_name"],
            "cve_id"        : cve_id,
            "technique_id"  : row["technique_id"],
            "technique_name": row["technique_name"],
            "source"        : "MITRE ATT&CK: technique description text",
        })

    # Source C: scan the relationship description (actor→technique link)
    if group_stix_id:
        rel_desc = rel_desc_lookup.get((group_stix_id, technique_stix_id), "")
        for cve_id in extract_cves_from_string(rel_desc):
            mitre_cve_rows.append({
                "group_id"      : row["group_id"],
                "actor_name"    : row["actor_name"],
                "cve_id"        : cve_id,
                "technique_id"  : row["technique_id"],
                "technique_name": row["technique_name"],
                "source"        : "MITRE ATT&CK: actor-technique relationship description",
            })

# Deduplicate on (group_id, cve_id) — keep the highest-confidence source
# Priority order: relationship description > technique description > external_reference
SOURCE_PRIORITY = {
    "MITRE ATT&CK: actor-technique relationship description": 0,
    "MITRE ATT&CK: technique description text":              1,
    "MITRE ATT&CK: technique external_reference":            2,
}

mitre_cve_df = pd.DataFrame(mitre_cve_rows)

if len(mitre_cve_df) > 0:
    mitre_cve_df["source_priority"] = mitre_cve_df["source"].map(SOURCE_PRIORITY)
    mitre_cve_df = (
        mitre_cve_df
        .sort_values("source_priority")
        .drop_duplicates(subset=["group_id", "cve_id"])
        .drop(columns="source_priority")
        .reset_index(drop=True)
    )

mitre_actor_cve = mitre_cve_df
mitre_actor_cve.to_csv(out_path("actor_cve_mitre.csv"), index=False)

# Summary broken down by source so you can see where data is coming from
print(f"\nMITRE-sourced actor-CVE links : {len(mitre_actor_cve):,} rows")
print(f"Actors with MITRE CVE data    : {mitre_actor_cve['group_id'].nunique():,}")
print(f"Distinct CVEs from MITRE      : {mitre_actor_cve['cve_id'].nunique():,}")
if len(mitre_actor_cve) > 0:
    print("\nBreakdown by source:")
    for source, count in mitre_actor_cve["source"].value_counts().items():
        print(f"  {count:>5,}  {source}")


Actor master dataset rows: 174
Columns: ['group_id', 'actor_name', 'aliases', 'actor_type', 'country_attribution', 'targeted_sectors', 'mitre_description', 'technique_count', 'tactic_count', 'technique_diversity_score']

MITRE-sourced actor-CVE links : 371 rows
Actors with MITRE CVE data    : 97
Distinct CVEs from MITRE      : 118

Breakdown by source:
    180  MITRE ATT&CK: actor-technique relationship description
    116  MITRE ATT&CK: technique description text
     75  MITRE ATT&CK: technique external_reference


## Cell 10: Save Supporting Mapping Tables

Expands the multi-value columns (aliases, countries, sectors) into separate
lookup tables — one row per value. These are used for filtering and joining
in analysis in other code blocks.

Files saved:
- `actor_master_dataset_mitre_stage.csv` — full actor table at this stage
- `actor_alias_mapping.csv` — one alias per row
- `actor_country_mapping.csv` — one country per row
- `actor_sector_mapping.csv` — one sector per row
- `actor_technique_mapping.csv` — one technique per row

In [ ]:
alias_rows, country_rows, sector_rows = [], [], []

for _, row in actor_master_dataset.iterrows():

    for alias in split_list(row["aliases"]):
        alias_rows.append({"group_id": row["group_id"], "alias": alias})

    for country in split_list(row["country_attribution"]):
        if country and country != "unknown":
            country_rows.append({
                "group_id": row["group_id"],
                "country" : country,
                "source"  : "MITRE description keyword extraction",
            })

    for sector in split_list(row["targeted_sectors"]):
        if sector and sector != "unknown":
            sector_rows.append({
                "group_id": row["group_id"],
                "sector"  : sector,
                "source"  : "MITRE description keyword extraction",
            })

actor_alias_mapping   = pd.DataFrame(alias_rows).drop_duplicates()
actor_country_mapping = pd.DataFrame(country_rows).drop_duplicates()
actor_sector_mapping  = pd.DataFrame(sector_rows).drop_duplicates()

# Save all MITRE-stage files
actor_master_dataset.to_csv(out_path("actor_master_dataset_mitre_stage.csv"), index=False)
actor_alias_mapping.to_csv(out_path("actor_alias_mapping.csv"),               index=False)
actor_country_mapping.to_csv(out_path("actor_country_mapping.csv"),           index=False)
actor_sector_mapping.to_csv(out_path("actor_sector_mapping.csv"),             index=False)
actor_technique_mapping.to_csv(out_path("actor_technique_mapping.csv"),       index=False)

print("MITRE-stage files saved to output folder.")
print(f"  Aliases:    {len(actor_alias_mapping):,} rows")
print(f"  Countries:  {len(actor_country_mapping):,} rows")
print(f"  Sectors:    {len(actor_sector_mapping):,} rows")
print(f"  Techniques: {len(actor_technique_mapping):,} rows")


MITRE-stage files saved to output folder.
  Aliases:    592 rows
  Countries:  323 rows
  Sectors:    436 rows
  Techniques: 4,546 rows


## Cell 11: OTX API Key Setup

Reads the AlienVault OTX API key `OTX_API_KEY` from Colab Secrets
The key is never printed or stored in the notebook.  
This cell is skipped automatically if `RUN_OTX = False` in Cell 3


In [ ]:
OTX_API_KEY = None
HEADERS     = {}

if RUN_OTX:
    try:
        OTX_API_KEY = userdata.get("OTX_API_KEY")
        if not OTX_API_KEY or not OTX_API_KEY.strip():
            raise ValueError("OTX_API_KEY secret is empty.")
        OTX_API_KEY = OTX_API_KEY.strip()
        HEADERS = {
            "X-OTX-API-KEY": OTX_API_KEY,
            "User-Agent"   : "praxis-cti-pipeline/1.0",
        }
        print("OTX API key loaded from Colab Secrets.")
    except Exception as e:
        print(f" Could not load OTX API key: {e}")
        print("   Add a secret named OTX_API_KEY via the key icon in the sidebar,")
        print("   or set RUN_OTX = False in Cell 3 to skip OTX.")
        RUN_OTX = False   # Disable OTX gracefully rather than crashing later
else:
    print("RUN_OTX = False — OTX step will be skipped.")


RUN_OTX = False — OTX step will be skipped.


## Cell 12: Search AlienVault OTX for Actor to CVE Mappings

For each actor in the master dataset, Alienvault OTX pulse data (CTI reports)
are searched using the actor's name and aliases. Any CVE identifiers found in matching pulses are extracted and saved as candidates for review.

**Review Process:**
- After cell runs, open actor_cve_mapping_candidates_otx.csv, review each row, and decide whether the pulse establishes a link between actor and CVE.
- For either those confirmed (or all if you dont want to do the manual review), copy the group_id and cve_id columns into a new file named actor_cve_mapping.csv and save it to input folder.

Files saved:
- `actor_cve_mapping_candidates_otx.csv` — CVE candidates with pulse evidence
- `otx_actor_search_log.csv` — record of every query run and results count

This cell is skipped if `RUN_OTX = False` or if the API key failed to load.


In [ ]:
# OTX resilience settings
# OTX can be slow to respond. These settings control how the code handles that.
OTX_CONNECT_TIMEOUT = 20   # seconds to wait while establishing the connection
OTX_READ_TIMEOUT    = 90    # seconds to wait for OTX to send back a response
OTX_MAX_RETRIES     = 3     # how many times to retry a timed-out or failed request
OTX_RETRY_BACKOFF   = [5, 15, 30]  # seconds to wait before each retry attempt

# OTX helper functions
CVE_PATTERN = re.compile(r"CVE-\d{4}-\d{4,7}", re.IGNORECASE)

def extract_cves_from_text(text):
    """Pull every CVE ID out of a text string. Returns a sorted, deduplicated list."""
    if text is None or pd.isna(text):
        return []
    return sorted(set(x.upper() for x in CVE_PATTERN.findall(str(text))))

def safe_text(value):
    """Safely convert any value to a plain string for CVE extraction."""
    if value is None:
        return ""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def pulse_to_text(pulse):
    """
    Combine all text fields of a pulse into one string for CVE scanning.
    Includes name, description, tags, references, adversary, and indicators.
    """
    fields = [
        safe_text(pulse.get(k, ""))
        for k in ["name", "description", "tags", "references",
                  "adversary", "industries", "targeted_countries"]
    ]
    for indicator in pulse.get("indicators", []):
        fields.append(safe_text(indicator))
    return "\n".join(fields)

def pulse_url(pulse):
    """Return the OTX web URL for a pulse."""
    pulse_id = pulse.get("id") or pulse.get("pulse_id")
    return f"https://otx.alienvault.com/pulse/{pulse_id}" if pulse_id else ""

def otx_get_with_retry(url, headers, max_retries=OTX_MAX_RETRIES):
    """
    GET a URL with automatic retry on timeout or server errors.

    Retry schedule:
      - ReadTimeout / ConnectTimeout: wait OTX_RETRY_BACKOFF[attempt] seconds, then retry
      - HTTP 429 (rate limit):        wait 30 seconds, then retry
      - HTTP 5xx (server error):      wait 10 seconds, then retry
      - HTTP 401/403 (auth failure):  raise immediately — retrying won't help

    Returns a requests.Response object on success.
    Raises the last exception if all retries are exhausted.
    """
    timeout = (OTX_CONNECT_TIMEOUT, OTX_READ_TIMEOUT)
    last_exc = None

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)

            # Auth failures are permanent — stop immediately
            if r.status_code in [401, 403]:
                raise PermissionError(
                    f"OTX API authentication failed (HTTP {r.status_code}). "
                    "Check your OTX_API_KEY secret."
                )

            # Rate limit — wait longer and retry
            if r.status_code == 429:
                wait = 30
                print(f"   OTX rate limit (HTTP 429) — waiting {wait}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait)
                continue

            # Server errors — short wait and retry
            if r.status_code >= 500:
                wait = 10
                print(f"   OTX server error (HTTP {r.status_code}) — waiting {wait}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r   # Success

        except requests.exceptions.Timeout as e:
            last_exc = e
            wait = OTX_RETRY_BACKOFF[min(attempt, len(OTX_RETRY_BACKOFF) - 1)]
            print(f"   OTX timeout on attempt {attempt + 1}/{max_retries} — waiting {wait}s before retry...")
            time.sleep(wait)

        except requests.exceptions.ConnectionError as e:
            last_exc = e
            wait = OTX_RETRY_BACKOFF[min(attempt, len(OTX_RETRY_BACKOFF) - 1)]
            print(f"  OTX connection error on attempt {attempt + 1}/{max_retries} — waiting {wait}s before retry...")
            time.sleep(wait)

    # All retries exhausted
    print(f" OTX request failed after {max_retries} attempts. Skipping query.")
    raise last_exc

def otx_search_pulses(query, max_pages=1):
    """
    Search OTX for pulses matching a query string.
    Handles timeouts, rate limits, and server errors with automatic retry.
    Returns a flat list of pulse objects across all pages fetched.
    On repeated failure, returns an empty list so the pipeline continues.
    """
    url = f"https://otx.alienvault.com/api/v1/search/pulses?q={quote(query)}"
    all_results = []

    for page_num in range(max_pages):
        try:
            r = otx_get_with_retry(url, HEADERS)
        except Exception as e:
            print(f"   Skipping query \"{query}\" page {page_num + 1} after repeated failures: {type(e).__name__}")
            break   # Move on to the next query rather than crashing the whole search

        data = r.json()
        results = data.get("results", [])
        if isinstance(results, list):
            all_results.extend(results)

        # Follow pagination if OTX provides a next-page URL
        next_url = data.get("next")
        if not next_url:
            break
        url = (next_url if str(next_url).startswith("http")
               else "https://otx.alienvault.com" + str(next_url))
        time.sleep(REQUEST_DELAY_SECONDS)

    return all_results

# Run OTX search
if RUN_OTX:
    print(f"Starting OTX search (connect timeout={OTX_CONNECT_TIMEOUT}s, read timeout={OTX_READ_TIMEOUT}s, max retries={OTX_MAX_RETRIES})...")

    actors_to_search = actor_master_dataset[["group_id", "actor_name", "aliases"]].copy()

    if MAX_ACTORS_TO_SEARCH is not None:
        actors_to_search = actors_to_search.head(MAX_ACTORS_TO_SEARCH)
        print(f"Searching {MAX_ACTORS_TO_SEARCH} actors (of {len(actor_master_dataset):,} total).")
    else:
        print(f"Searching all {len(actor_master_dataset):,} actors.")

    candidate_rows = []
    search_log     = []
    skipped_queries = 0

    for _, actor in actors_to_search.iterrows():
        group_id   = actor["group_id"]
        actor_name = actor["actor_name"]
        aliases    = split_list(actor["aliases"])

        # Build a deduplicated query list: actor name first, then aliases
        query_terms = list(dict.fromkeys([actor_name] + aliases))[:MAX_ALIASES_PER_ACTOR]
        print(f"  {group_id} | {actor_name}: querying {query_terms}")

        for query in query_terms:
            try:
                pulses = otx_search_pulses(query, max_pages=MAX_PAGES_PER_QUERY)
                pulse_count = len(pulses)
            except Exception as e:
                # otx_search_pulses already printed the error; just log and continue
                pulses = []
                pulse_count = 0
                skipped_queries += 1

            search_log.append({
                "group_id"              : group_id,
                "actor_name"            : actor_name,
                "query"                 : query,
                "pulse_results_returned": pulse_count,
            })

            for pulse in pulses:
                cves = extract_cves_from_text(pulse_to_text(pulse))
                for cve in cves:
                    candidate_rows.append({
                        "review_status"  : "needs_review",
                        "group_id"       : group_id,
                        "actor_name"     : actor_name,
                        "matched_query"  : query,
                        "cve_id"         : cve,
                        "pulse_id"       : pulse.get("id", pulse.get("pulse_id", "")),
                        "pulse_name"     : pulse.get("name", ""),
                        "pulse_url"      : pulse_url(pulse),
                        "evidence_source": "AlienVault OTX pulse search",
                        "notes"          : ("Candidate only — approve only if the pulse "
                                            "clearly links this actor to this CVE."),
                    })

            time.sleep(REQUEST_DELAY_SECONDS)

    otx_candidates = pd.DataFrame(candidate_rows).drop_duplicates()
    otx_search_log = pd.DataFrame(search_log)

    # Cross-reference candidates with CISA KEV to surface high-priority CVEs
    if len(otx_candidates) > 0 and os.path.exists(CISA_KEV_FILE):
        kev_ref = pd.read_csv(CISA_KEV_FILE)
        if "cveID" in kev_ref.columns:
            kev_ref = kev_ref.rename(columns={"cveID": "cve_id"})
        kev_ref["cve_id"] = kev_ref["cve_id"].str.upper()
        kev_cols = [c for c in ["cve_id", "vendorProject", "product",
                                 "vulnerabilityName", "dateAdded",
                                 "knownRansomwareCampaignUse"]
                    if c in kev_ref.columns]
        otx_candidates = otx_candidates.merge(kev_ref[kev_cols], on="cve_id", how="left")
        otx_candidates["is_kev"] = otx_candidates["dateAdded"].notna().astype(int)

    otx_candidates.to_csv(out_path("actor_cve_mapping_candidates_otx.csv"), index=False)
    otx_search_log.to_csv(out_path("otx_actor_search_log.csv"),             index=False)

    print(f"\nOTX search complete.")
    print(f"  CVE candidates found: {len(otx_candidates):,}")
    print(f"  Queries run:          {len(otx_search_log):,}")
    print(f"  Queries skipped (persistent failure): {skipped_queries}")
    print(f"  Saved to output folder.")

else:
    print("RUN_OTX = False — OTX search skipped.")
    otx_candidates = pd.DataFrame()



Starting OTX search (connect timeout=20s, read timeout=90s, max retries=3)...
Searching all 174 actors.
  G0001 | Axiom: querying ['Axiom', 'Group 72']
  G0002 | Moafee: querying ['Moafee']
  G0003 | Cleaver: querying ['Cleaver', 'Threat Group 2889', 'TG-2889']
  G0004 | Ke3chang: querying ['Ke3chang', 'APT15', 'Mirage', 'Vixen Panda', 'GREF', 'Playful Dragon']
  G0005 | APT12: querying ['APT12', 'IXESHE', 'DynCalc', 'Numbered Panda', 'DNSCALC']
   OTX server error (HTTP 502) — waiting 10s before retry 1/3...
  G0006 | APT1: querying ['APT1', 'Comment Crew', 'Comment Group', 'Comment Panda']
  G0007 | APT28: querying ['APT28', 'IRON TWILIGHT', 'SNAKEMACKEREL', 'Swallowtail', 'Group 74', 'Sednit']
  G0008 | Carbanak: querying ['Carbanak', 'Anunak']
  G0009 | Deep Panda: querying ['Deep Panda', 'Shell Crew', 'WebMasters', 'KungFu Kittens', 'PinkPanther', 'Black Vine']
  G0010 | Turla: querying ['Turla', 'IRON HUNTER', 'Group 88', 'Waterbug', 'WhiteBear', 'Snake']
  G0011 | PittyTiger: qu

## Cell 13: Process NVD Bulk Feed Files

This cell reads the National Vulnerability Database (NVD) files and extracts a `(cve_id, cvss_score)` table. CVSS scores are used later to calculate `avg_cvss_score`, `max_cvss_score`, and `critical_vuln_ratio` per actor.

Output saved: `nvd_cvss.csv`


In [ ]:
#13 Download NVD data to NVD_full.json
# Load NVD API key from Colab Secrets
try:
    NVD_API_KEY = userdata.get("NVD_API_KEY")
    if not NVD_API_KEY or not NVD_API_KEY.strip():
        raise ValueError("empty")
    NVD_API_KEY = NVD_API_KEY.strip()
    print(" NVD API key loaded from Colab Secrets")
except Exception:
    NVD_API_KEY = None
    print(" No NVD_API_KEY secret found — running without key (slower)")


NVD_OUT_FILE  = os.path.join(INPUT_DIR, "nvd_full.json")
NVD_PAGE_SIZE = 2000

nvd_headers = {"User-Agent": "praxis-cti-pipeline/1.0"}
if NVD_API_KEY:
    nvd_headers["apiKey"] = NVD_API_KEY

#  Skip download if file already exists
# nvd_full.json is large (~300MB) and takes time to download.
# If it already exists, this cell skips the download automatically.
# To force a fresh download, delete nvd_full.json from the input folder and rerun this cell

if os.path.exists(NVD_OUT_FILE):
    size_mb = os.path.getsize(NVD_OUT_FILE) / 1_048_576
    print(f" nvd_full.json already exists ({size_mb:.0f} MB) — skipping download.")
    print(f" Delete it from the input folder and re-run to get a fresh copy.")
    print(f" Proceed to Cell 13b to process this file into nvd_cvss.")

else:
    print(f"nvd_full.json not found — starting download...")

    def fetch_nvd_all_cves():
        all_vulns = []
        start     = 0
        total     = None

        while True:
            url = (f"https://services.nvd.nist.gov/rest/json/cves/2.0"
                   f"?startIndex={start}&resultsPerPage={NVD_PAGE_SIZE}")

            for attempt in range(5):
                try:
                    r = requests.get(url, headers=nvd_headers, timeout=60)
                    if r.status_code == 403:
                        raise PermissionError("NVD API key rejected or rate limit hit.")
                    if r.status_code == 429:
                        print(f"\n  Rate limited — waiting 35s...")
                        time.sleep(35)
                        continue
                    r.raise_for_status()
                    break
                except requests.exceptions.Timeout:
                    wait = [10, 20, 30, 60, 120][attempt]
                    print(f"\n  Timeout on attempt {attempt+1}/5 — waiting {wait}s...")
                    time.sleep(wait)

            data = r.json()

            if total is None:
                total = data["totalResults"]
                rate  = "50 req/30s (with API key)" if NVD_API_KEY else "5 req/30s (no key)"
                print(f"Total CVEs in NVD : {total:,}")
                print(f"Pages to fetch    : {-(-total // NVD_PAGE_SIZE):,}")
                print(f"Rate limit        : {rate}")

            batch = data.get("vulnerabilities", [])
            all_vulns.extend(batch)
            start += len(batch)
            print(f"  Fetched {start:,} / {total:,} CVEs...", end="\r")

            if start >= total:
                break

            time.sleep(0.7 if NVD_API_KEY else 6.5)

        print(f"\nWriting {len(all_vulns):,} CVEs to {NVD_OUT_FILE}...")
        output = {
            "resultsPerPage" : len(all_vulns),
            "startIndex"     : 0,
            "totalResults"   : len(all_vulns),
            "format"         : "NVD_CVE",
            "version"        : "2.0",
            "timestamp"      : pd.Timestamp.now().isoformat(),
            "vulnerabilities": all_vulns,
        }
        with open(NVD_OUT_FILE, "w", encoding="utf-8") as f:
            json.dump(output, f)

        size_mb = os.path.getsize(NVD_OUT_FILE) / 1_048_576
        print(f" Saved: {NVD_OUT_FILE} ({size_mb:.0f} MB)")
        print(f" Proceed to Cell 13b to process this file into nvd_cvss.")

    fetch_nvd_all_cves()

 NVD API key loaded from Colab Secrets
 nvd_full.json already exists (1459 MB) — skipping download.
 Delete it from the input folder and re-run to get a fresh copy.
 Proceed to Cell 13b to process this file into nvd_cvss.


In [ ]:
# 13B  Process NVD data into a clean CVE → CVSS score lookup
# Reads nvd_full.json (or any other NVD JSON/ZIP files in the input folder),
# extracts cve_id and cvss_score columns, and saves nvd_cvss.csv.
# Also updates the nvd_cvss variable in memory used by Cell 16.
#
# Re-run this cell any time you:
#   - Run Cell 13 for the first time
#   - Replace nvd_full.json with a fresh download
#   - Add additional NVD feed files to the input folder

def find_nvd_files():
    """Find all NVD JSON/ZIP files in the input folder."""
    patterns = ["*nvd*.json", "*nvd*.zip", "*nvdcve*.json", "*nvdcve*.zip",
                "*CVE-*.json", "*CVE-*.zip", "*cve*.json", "*cve*.zip"]
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(INPUT_DIR, p)))
    files = [f for f in files if os.path.isfile(f)]
    return list(dict.fromkeys(files))


def load_json_objects_from_file(path):
    """
    Load JSON from a plain .json file or a .zip archive containing .json files.
    """
    if path.lower().endswith(".zip"):
        json_objects = []
        with zipfile.ZipFile(path, "r") as z:
            for name in z.namelist():
                if name.lower().endswith(".json"):
                    with z.open(name) as f:
                        json_objects.append(json.load(f))
        return json_objects
    with open(path, "r", encoding="utf-8") as f:
        return [json.load(f)]


def extract_cvss_score(cve_obj):
    """
    Extract the best available CVSS base score from an NVD CVE object.

    Handles all known NVD format variations:
      - cvssMetricV40, cvssMetricV31, cvssMetricV30, cvssMetricV3, cvssMetricV2
      - baseScore inside cvssData (standard) or directly on the metric entry (some exports)
      - Prefers NVD Primary score over CNA Secondary score when both present
    """
    metrics = cve_obj.get("metrics", {})
    if not metrics:
        return np.nan

    for key in ["cvssMetricV40", "cvssMetricV31", "cvssMetricV30",
                "cvssMetricV3", "cvssMetricV2"]:
        entries = metrics.get(key)
        if not entries:
            continue
        primary   = [e for e in entries if e.get("type", "Primary") == "Primary"]
        to_search = primary if primary else entries
        for entry in to_search:
            score = entry.get("cvssData", {}).get("baseScore")
            if score is not None:
                return float(score)
            score = entry.get("baseScore")
            if score is not None:
                return float(score)
    return np.nan


def process_nvd_files():
    """
    Parse all NVD files in the input folder into a cve_id / cvss_score table.
    Supports NVD API 2.0 format (vulnerabilities key) and
    NVD legacy 1.1 format (CVE_Items key).
    """
    nvd_files = find_nvd_files()

    if not nvd_files:
        print("  No NVD files found in input folder.")
        print("   Run Cell 13 first to download nvd_full.json.")
        empty = pd.DataFrame(columns=["cve_id", "cvss_score"])
        empty.to_csv(out_path("nvd_cvss.csv"), index=False)
        return empty

    print(f"Found {len(nvd_files)} NVD file(s):")
    for f in nvd_files:
        size_mb = os.path.getsize(f) / 1_048_576
        print(f"  {os.path.basename(f)} ({size_mb:.0f} MB)")

    rows = []
    for path in nvd_files:
        print(f"\nProcessing: {os.path.basename(path)}")
        for data in load_json_objects_from_file(path):

            top_keys = list(data.keys())
            print(f"  Top-level keys      : {top_keys}")

            # NVD API 2.0 format
            if "vulnerabilities" in data:
                items = data["vulnerabilities"]
                print(f"  Format              : NVD API 2.0")
                print(f"  CVE entries         : {len(items):,}")
                if items:
                    sample_metrics = items[0].get("cve", {}).get("metrics", {})
                    print(f"  Metric keys (first CVE): {list(sample_metrics.keys())}")

                for item in items:
                    cve    = item.get("cve", {})
                    cve_id = cve.get("id")
                    if cve_id:
                        rows.append({
                            "cve_id"     : cve_id.upper(),
                            "cvss_score" : extract_cvss_score(cve),
                        })

            # ── NVD legacy 1.1 format ──────────────────────────────────────────
            elif "CVE_Items" in data:
                items = data["CVE_Items"]
                print(f"  Format              : NVD legacy 1.1")
                print(f"  CVE entries         : {len(items):,}")

                for item in items:
                    cve_id = (item.get("cve", {})
                                  .get("CVE_data_meta", {})
                                  .get("ID"))
                    impact     = item.get("impact", {})
                    cvss_score = np.nan
                    if "baseMetricV3" in impact:
                        cvss_score = (impact["baseMetricV3"]
                                           .get("cvssV3", {})
                                           .get("baseScore", np.nan))
                    elif "baseMetricV2" in impact:
                        cvss_score = (impact["baseMetricV2"]
                                           .get("cvssV2", {})
                                           .get("baseScore", np.nan))
                    if cve_id:
                        rows.append({
                            "cve_id"     : cve_id.upper(),
                            "cvss_score" : cvss_score,
                        })

            else:
                print(f"   Unrecognised format — skipping. Keys: {top_keys}")

    if not rows:
        print("\n No CVE rows extracted ")
        empty = pd.DataFrame(columns=["cve_id", "cvss_score"])
        empty.to_csv(out_path("nvd_cvss.csv"), index=False)
        return empty

    nvd_cvss = pd.DataFrame(rows).drop_duplicates(subset=["cve_id"])

    has_score   = nvd_cvss["cvss_score"].notna().sum()
    no_score    = nvd_cvss["cvss_score"].isna().sum()
    pct_missing = no_score / len(nvd_cvss) * 100

    print(f"\nNVD processing complete:")
    print(f"  Total CVEs loaded    : {len(nvd_cvss):,}")
    print(f"  CVEs with CVSS score : {has_score:,}")
    print(f"  CVEs without score   : {no_score:,} ({pct_missing:.1f}%)")
    if pct_missing > 50:
        print(f"    More than 50% missing — check metric keys in diagnostic above.")

    nvd_cvss.to_csv(out_path("nvd_cvss.csv"), index=False)
    print(f"\n Saved nvd_cvss.csv to output folder.")
    print(f" Re-run Cell 16 to calculate vulnerability features with updated scores.")
    return nvd_cvss


nvd_cvss = process_nvd_files()

Found 2 NVD file(s):
  nvdcve-2.0-modified.json (10 MB)
  nvd_full.json (1459 MB)

Processing: nvdcve-2.0-modified.json
  Top-level keys      : ['resultsPerPage', 'startIndex', 'totalResults', 'format', 'version', 'timestamp', 'vulnerabilities']
  Format              : NVD API 2.0
  CVE entries         : 2,400
  Metric keys (first CVE): ['cvssMetricV2']

Processing: nvd_full.json
  Top-level keys      : ['resultsPerPage', 'startIndex', 'totalResults', 'format', 'version', 'timestamp', 'vulnerabilities']
  Format              : NVD API 2.0
  CVE entries         : 350,896
  Metric keys (first CVE): ['cvssMetricV2']

NVD processing complete:
  Total CVEs loaded    : 350,896
  CVEs with CVSS score : 331,198
  CVEs without score   : 19,698 (5.6%)

 Saved nvd_cvss.csv to output folder.
 Re-run Cell 16 to calculate vulnerability features with updated scores.


## Cell 14: Load & Process CISA KEV Data

This code loads the CISA Known Exploited Vulnerabilities catalog. The KEV list flags CVEs that CISA has confirmed are being actively exploited in the wild.

Key features derived:
- `kev_count` — how many of an actor's associated CVEs are in the KEV list
- `kev_ratio` — proportion of the actor's CVEs that are KEV-listed

Output saved: `cisa_kev_processed.csv`


In [ ]:
def load_cisa_kev():
    """
    Load the CISA KEV CSV, normalise the CVE ID column name,
    and add an is_kev flag column.
    Returns an empty DataFrame if the file is missing.
    """
    if not os.path.exists(CISA_KEV_FILE):
        print("CISA KEV file not found — KEV features will be zero.")
        return pd.DataFrame(columns=["cve_id", "is_kev"])

    kev = pd.read_csv(CISA_KEV_FILE)

    # CISA exports use 'cveID'; normalise to 'cve_id' for consistent merging
    if "cveID" in kev.columns:
        kev = kev.rename(columns={"cveID": "cve_id"})

    if "cve_id" not in kev.columns:
        raise ValueError("CISA KEV file must contain a 'cveID' or 'cve_id' column.")

    kev["cve_id"] = kev["cve_id"].str.upper()
    kev["is_kev"] = 1   # every row in the KEV file is a known exploited vuln

    kev.to_csv(out_path("cisa_kev_processed.csv"), index=False)
    print(f"CISA KEV loaded: {len(kev):,} entries")
    return kev

kev = load_cisa_kev()


CISA KEV loaded: 1,590 entries


## Cell 15: Load Reviewed Actor-to-CVE Mapping

This code builds the final actor-to-CVE table used for all vulnerability feature
engineering in Cell 16. It combines two sources from the output folder that previous code created:

| Source | How it's produced |
|---|---|
| `actor_cve_mitre.csv` | Extracted in Cell 9 from MITRE ATT&CK data |
| `actor_cve_mapping.csv` | Created from Alienvault OTX data in Cell 12 |

If `actor_cve_mapping.csv` does not exist yet (first run before any OTX review),
the pipeline uses MITRE CVEs only. All vulnerability features will be zero for
actors that appear in neither source.


In [ ]:
def load_actor_cve_mapping():
    """
    Load the OTX actor-to-CVE mapping from the input folder.
    Returns an empty DataFrame with the correct columns if the file is missing.
    """
    if not os.path.exists(ACTOR_CVE_MAPPING_FILE):
        print("actor_cve_mapping.csv not found in input folder.")
        print("Only MITRE-sourced CVEs will be used for vulnerability features.")
        print("Populate this file with reviewed OTX candidates to broaden coverage.")
        return pd.DataFrame(columns=["group_id", "cve_id"])

    df = pd.read_csv(ACTOR_CVE_MAPPING_FILE)

    if "group_id" not in df.columns or "cve_id" not in df.columns:
        raise ValueError(
            "actor_cve_mapping.csv must contain 'group_id' and 'cve_id' columns."
        )

    df = df[["group_id", "cve_id"]].dropna().drop_duplicates()
    df["cve_id"] = df["cve_id"].str.upper()
    print(f"Reviewed OTX actor-CVE mappings loaded: {len(df):,} rows")
    return df


# Load both sources
otx_reviewed  = load_actor_cve_mapping()
mitre_cve_for_scoring = mitre_actor_cve[["group_id", "cve_id"]].copy()

# Combine: MITRE CVEs +  OTX CVEs
reviewed_actor_cve = (
    pd.concat([mitre_cve_for_scoring, otx_reviewed], ignore_index=True)
    .drop_duplicates(subset=["group_id", "cve_id"])
)

print(f"\nCombined actor-CVE table:")
print(f"  From MITRE ATT&CK (auto) : {len(mitre_cve_for_scoring):,} rows")
print(f"  From reviewed OTX        : {len(otx_reviewed):,} rows")
print(f"  Combined (deduplicated)  : {len(reviewed_actor_cve):,} rows")
print(f"  Actors covered           : {reviewed_actor_cve['group_id'].nunique():,}")

Reviewed OTX actor-CVE mappings loaded: 108 rows

Combined actor-CVE table:
  From MITRE ATT&CK (auto) : 371 rows
  From reviewed OTX        : 108 rows
  Combined (deduplicated)  : 464 rows
  Actors covered           : 112


## Cell 16: Engineer Vulnerability Features

Joins the actor-CVE mapping with NVD CVSS scores and the CISA KEV
flag to build per-actor vulnerability features:

| Feature | Description |
|---|---|
| `associated_cve_count` | Total distinct CVEs linked to this actor |
| `avg_cvss_score` | Average CVSS base score across associated CVEs |
| `max_cvss_score` | Highest CVSS score among associated CVEs |
| `kev_count` | Number of associated CVEs on the CISA KEV list |
| `kev_ratio` | Fraction of associated CVEs that are KEV-listed |
| `critical_vuln_ratio` | Fraction of CVEs with CVSS ≥ 9.0 |

These feed into the Capability Score and Activity Score in Cell 18.

Outputs saved: `actor_vulnerability_detail.csv`, `actor_vulnerability_features.csv`


In [ ]:
if len(reviewed_actor_cve) > 0:

    # Join CVEs with KEV flags and CVSS scores
    actor_vuln = reviewed_actor_cve.merge(
        kev[["cve_id", "is_kev"]], on="cve_id", how="left"
    )
    actor_vuln = actor_vuln.merge(
        nvd_cvss[["cve_id", "cvss_score"]], on="cve_id", how="left"
    )

    actor_vuln["is_kev"]        = actor_vuln["is_kev"].fillna(0).astype(int)
    actor_vuln["cvss_score"]    = pd.to_numeric(actor_vuln["cvss_score"], errors="coerce")

    # Flag CVEs with CVSS >= 9.0 as critical
    actor_vuln["critical_flag"] = (actor_vuln["cvss_score"] >= 9.0).astype(int)

    # Aggregate to one row per actor
    vuln_features = actor_vuln.groupby("group_id").agg(
        associated_cve_count  = ("cve_id",        "nunique"),
        avg_cvss_score        = ("cvss_score",     "mean"),
        max_cvss_score        = ("cvss_score",     "max"),
        kev_count             = ("is_kev",         "sum"),
        critical_vuln_ratio   = ("critical_flag",  "mean"),
    ).reset_index()

    # KEV ratio = what fraction of the actor's CVEs are known-exploited
    vuln_features["kev_ratio"] = (
        vuln_features["kev_count"] / vuln_features["associated_cve_count"]
    )

    actor_vuln.to_csv(out_path("actor_vulnerability_detail.csv"), index=False)

else:
    # No reviewed mappings yet — create empty feature table
    vuln_features = pd.DataFrame(columns=[
        "group_id", "associated_cve_count", "avg_cvss_score", "max_cvss_score",
        "kev_count", "critical_vuln_ratio", "kev_ratio",
    ])

vuln_features.to_csv(out_path("actor_vulnerability_features.csv"), index=False)
print(f"Vulnerability features saved for {len(vuln_features):,} actors.")


Vulnerability features saved for 112 actors.


## Cell 17: Process ACLED Political Violence Data

This code loads the ACLED monthly political violence summary from `acled.xlsx` and computes country-level geopolitical instability indicators. The ACLED file must contain these columns (names are matched case-insensitively):

| Column | Meaning |
|---|---|
| COUNTRY | Country name |
| MONTH | Month (name or number) |
| YEAR | Four-digit year |
| EVENTS | Count of political violence events that month |

Country-level features calculated:

| Feature | Description |
|---|---|
| `acled_political_violence_events_total` | Sum of all monthly event counts |
| `acled_months_observed` | Number of months of data available |
| `acled_avg_monthly_political_violence_events` | Average events per month |
| `acled_max_monthly_political_violence_events` | Peak month event count |
| `geo_instability_score` | Normalized total events (0–1) |
| `conflict_intensity_index` | Normalized average monthly events (0–1) |

These two normalized scores feed into the Opportunity Score in Cell 18.

Output saved: `acled_country_political_violence_features.csv`


In [ ]:
def standardize_acled_columns(acled):
    """
    Normalise ACLED column names so the rest of the code works regardless of
    minor variations in the export (e.g. 'Events' vs 'EVENTS' vs 'event count').
    Raises a helpful error if required columns are still missing after mapping.
    """
    original_columns = list(acled.columns)

    # Lower-case and strip whitespace from all column names
    acled.columns = [str(c).strip().lower() for c in acled.columns]

    rename_map = {}
    for col in acled.columns:
        if col == "country":
            rename_map[col] = "country"
        elif col == "month":
            rename_map[col] = "month"
        elif col == "year":
            rename_map[col] = "year"
        elif any(kw in col for kw in [
            "political violence", "number of political violence",
            "political_violence", "event count", "events"
        ]):
            rename_map[col] = "political_violence_events"

    acled = acled.rename(columns=rename_map)

    required = ["country", "month", "year", "political_violence_events"]
    missing  = [c for c in required if c not in acled.columns]
    if missing:
        print("Original ACLED columns:", original_columns)
        print("After standardisation:", list(acled.columns))
        raise ValueError(
            "ACLED file missing required columns after standardisation: "
            + ", ".join(missing)
        )
    return acled

def process_acled_political_violence():
    """
    Load the ACLED monthly political violence file, aggregate to country level,
    and compute normalized geopolitical instability features.
    """
    if not os.path.exists(ACLED_FILE):
        print("ACLED file not found — geopolitical features will be zero.")
        return pd.DataFrame(columns=[
            "country",
            "acled_political_violence_events_total",
            "acled_months_observed",
            "acled_avg_monthly_political_violence_events",
            "acled_max_monthly_political_violence_events",
            "geo_instability_score",
            "conflict_intensity_index",
        ])

    # Load (supports both .xlsx and .csv)
    if ACLED_FILE.lower().endswith((".xlsx", ".xls")):
        acled = pd.read_excel(ACLED_FILE)
    else:
        acled = pd.read_csv(ACLED_FILE)

    acled = standardize_acled_columns(acled)
    print(f"ACLED rows loaded: {len(acled):,}")

    # Clean values
    acled["country"]                  = acled["country"].astype(str).str.strip()

    print("\n--- Before numeric conversion ---")
    print(f"'month' column dtype: {acled['month'].dtype}")
    print("Value counts for 'month' (including NaNs):")
    print(acled['month'].value_counts(dropna=False))
    print(f"\n'year' column dtype: {acled['year'].dtype}")
    print("Value counts for 'year' (including NaNs):")
    print(acled['year'].value_counts(dropna=False))
    print("-----------------------------------")

    # Convert month names to numbers
    month_name_to_num = {
        'january': 1, 'february': 2, 'march': 3, 'april': 4,
        'may': 5, 'june': 6, 'july': 7, 'august': 8,
        'september': 9, 'october': 10, 'november': 11, 'december': 12
    }
    # Ensure month column is string type and lowercased for mapping
    acled['month'] = acled['month'].astype(str).str.lower().map(month_name_to_num)

    acled["month"]                    = pd.to_numeric(acled["month"],  errors="coerce")
    acled["year"]                     = pd.to_numeric(acled["year"],   errors="coerce")
    acled["political_violence_events"] = (
        pd.to_numeric(acled["political_violence_events"], errors="coerce").fillna(0)
    )

    print(f"\nShape before dropping NaNs: {acled.shape}")
    print(f"Unique countries before dropping NaNs: {acled['country'].nunique()}")
    print("ACLED DataFrame head before dropping NaNs:")
    display(acled.head())

    # Drop rows that are missing country, month, or year
    acled = acled.dropna(subset=["country", "month", "year"])

    print(f"Shape after dropping NaNs: {acled.shape}")

    # Aggregate from monthly rows → one row per country
    acled_features = acled.groupby("country").agg(
        acled_political_violence_events_total        = ("political_violence_events", "sum"),
        acled_months_observed                        = ("political_violence_events", "count"),
        acled_avg_monthly_political_violence_events  = ("political_violence_events", "mean"),
        acled_max_monthly_political_violence_events  = ("political_violence_events", "max"),
        acled_first_year                             = ("year", "min"),
        acled_last_year                              = ("year", "max"),
    ).reset_index()

    # Compatibility aliases (referenced in the merge step below)
    acled_features["acled_event_count"] = (
        acled_features["acled_political_violence_events_total"]
    )
    acled_features["acled_fatalities"] = 0   # not in this ACLED export

    # Geopolitical opportunity inputs
    # geo_instability_score:  normalized total political violence events
    #    captures countries with persistently high overall violence
    # conflict_intensity_index: normalized average monthly events
    #   captures countries with high per-month intensity
    acled_features["geo_instability_score"] = normalize_series(
        acled_features["acled_political_violence_events_total"]
    )
    acled_features["conflict_intensity_index"] = normalize_series(
        acled_features["acled_avg_monthly_political_violence_events"]
    )

    acled_features.to_csv(
        out_path("acled_country_political_violence_features.csv"), index=False
    )
    print(f"ACLED country features saved for {len(acled_features):,} countries.")
    return acled_features

acled_features = process_acled_political_violence()

ACLED rows loaded: 28,934

--- Before numeric conversion ---
'month' column dtype: object
Value counts for 'month' (including NaNs):
month
March        2487
April        2478
January      2473
May          2464
February     2460
August       2382
September    2379
June         2369
October      2369
November     2368
July         2360
December     2345
Name: count, dtype: int64

'year' column dtype: int64
Value counts for 'year' (including NaNs):
year
2021    2945
2020    2550
2019    1939
2018    1937
2022    1395
2023    1392
2024    1380
2025    1358
2017     928
2016     892
2015     733
2011     696
2012     696
2013     696
2014     696
2010     684
2008     576
1999     576
1997     576
2000     576
1998     576
2001     576
2004     576
2007     576
2006     576
2005     576
2002     576
2003     576
2009     576
2026     529
Name: count, dtype: int64
-----------------------------------

Shape before dropping NaNs: (28934, 4)
Unique countries before dropping NaNs: 250
ACLED Dat

,country,month,year,political_violence_events
0,Afghanistan,1,2017,865
1,Afghanistan,2,2017,697
2,Afghanistan,3,2017,1166
3,Afghanistan,4,2017,1079
4,Afghanistan,5,2017,1229


Shape after dropping NaNs: (28934, 4)
ACLED country features saved for 250 countries.


## Cell 17b: Merge ACLED Features to Actors

This code joins country-level instability features to actors via the
`actor_country_mapping` table built in Cell 10. Because an actor may be attributed to multiple countries, the pipeline takes the maximum score across all attributed countries — the actor operates in (or on behalf of) the most unstable country it is linked to.

Output saved: `actor_acled_features.csv`


In [ ]:
if len(actor_country_mapping) > 0 and len(acled_features) > 0:

    # Join country instability data to the actor-country lookup table
    actor_country_acled = actor_country_mapping.merge(
        acled_features, on="country", how="left"
    )

    # Aggregate to actor level: take the maximum score across attributed countries
    actor_acled_features = actor_country_acled.groupby("group_id").agg(
        acled_event_count                           = ("acled_event_count",                          "max"),
        # acled_fatalities                            = ("acled_fatalities",                           "max"),
        acled_political_violence_events_total       = ("acled_political_violence_events_total",      "max"),
        acled_months_observed                       = ("acled_months_observed",                      "max"),
        acled_avg_monthly_political_violence_events = ("acled_avg_monthly_political_violence_events","max"),
        acled_max_monthly_political_violence_events = ("acled_max_monthly_political_violence_events","max"),
        geo_instability_score                       = ("geo_instability_score",                      "max"),
        conflict_intensity_index                    = ("conflict_intensity_index",                   "max"),
    ).reset_index()

else:
    actor_acled_features = pd.DataFrame(columns=[
        "group_id", "acled_event_count", "acled_fatalities",
        "acled_political_violence_events_total", "acled_months_observed",
        "acled_avg_monthly_political_violence_events",
        "acled_max_monthly_political_violence_events",
        "geo_instability_score", "conflict_intensity_index",
    ])

actor_acled_features.to_csv(out_path("actor_acled_features.csv"), index=False)
print(f"Actor ACLED features saved for {len(actor_acled_features):,} actors.")


Actor ACLED features saved for 174 actors.


## Cell 18: Assemble the Final Actor Master Dataset

This code merges all engineered features into a single actor-level data set

```
actor_master_dataset          (MITRE metadata + technique/tactic counts)
    └── + vuln_features       (CVE, CVSS, KEV features)
    └── + actor_acled_features (geopolitical instability features)
```

Missing numeric values are filled with zero so every actor has a complete
feature vector even when data sources are unavailable.


In [ ]:
final_df = actor_master_dataset.copy()

# Merge vulnerability features (from reviewed actor-CVE mapping + NVD + KEV)
final_df = final_df.merge(vuln_features,        on="group_id", how="left")

# Merge geopolitical features (from ACLED via country attribution)
final_df = final_df.merge(actor_acled_features, on="group_id", how="left")

# Fill missing numeric columns with zero so scoring works on every row
numeric_defaults = {
    "associated_cve_count"                      : 0,
    "avg_cvss_score"                            : 0,
    "max_cvss_score"                            : 0,
    "kev_count"                                 : 0,
    "kev_ratio"                                 : 0,
    "critical_vuln_ratio"                       : 0,
    "acled_event_count"                         : 0,
    "acled_political_violence_events_total"     : 0,
    "acled_months_observed"                     : 0,
    "acled_avg_monthly_political_violence_events": 0,
    "acled_max_monthly_political_violence_events": 0,
    "geo_instability_score"                     : 0,
    "conflict_intensity_index"                  : 0,
}

for col, default in numeric_defaults.items():
    if col not in final_df.columns:
        final_df[col] = default
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce").fillna(default)

print(f"Final dataset assembled: {len(final_df):,} actors × {len(final_df.columns)} columns")


Final dataset assembled: 174 actors × 23 columns


## Cell 19: Calculate Composite Risk Scores

This code computes four scores for each actor using the assembled features.
All input components are normalized to 0–1 before weighting.

### Capability Score (how capable is the actor?)
Reflects the breadth and severity of the actor's documented attack toolkit and capability.

| Component | Weight | Source |
|---|---|---|
| Technique count (normalized) | 50% | MITRE ATT&CK |
| Average CVSS score (normalized) | 25% | NVD |
| KEV ratio (normalized) | 25% | CISA KEV |

### Opportunity Score (how favorable is the operating environment?)
Reflects the geopolitical instability of the countries the actor is attributed to to bring in real-world context to the activities.

| Component | Weight | Source |
|---|---|---|
| `geo_instability_score` (total violence, normalized) | 70% | ACLED |
| `conflict_intensity_index` (avg monthly violence, normalized) | 30% | ACLED |

### Activity Score (how active is the actor in exploiting vulnerabilities?)
Reflects the volume of confirmed exploitation activity.

| Component | Weight | Source |
|---|---|---|
| Associated CVE count (normalized) | 40% | Actor to CVE mapping |
| KEV count (normalized) | 60% | CISA KEV |

### Composite Risk Score
| Component | Weight |
|---|---|
| Capability Score | 50% |
| Opportunity Score | 20% |
| Activity Score | 30% |


In [ ]:
# Capability Score
# Measures the breadth and severity of an actor's documented attack toolkit.
# This is a score to show how technically capable the actor appears to be.
final_df["capability_score"] = (
    normalize_series(final_df["technique_count"]) * 0.50 +   # ATT&CK breadth
    normalize_series(final_df["avg_cvss_score"])  * 0.25 +   # CVE severity
    normalize_series(final_df["kev_ratio"])        * 0.25    # confirmed exploitation rate
)

# Opportunity Score
# Measures how unstable the geopolitical environment is for the actor's
# attributed countries. Already normalized in Cell 17 — no need to re-normalize.
final_df["opportunity_score"] = (
    final_df["geo_instability_score"]    * 0.70 +   # total violence volume
    final_df["conflict_intensity_index"] * 0.30     # per-month violence intensity
)

# Activity Score
# Measures how actively the actor exploits known vulnerabilities.
final_df["activity_score"] = (
    normalize_series(final_df["associated_cve_count"]) * 0.40 +  # CVE breadth
    normalize_series(final_df["kev_count"])            * 0.60    # confirmed exploitation count
)

# Composite Risk Score
final_df["composite_risk_score"] = (
    final_df["capability_score"]  * 0.50 +
    final_df["opportunity_score"] * 0.20 +
    final_df["activity_score"]    * 0.30
)

#  Binary data-completeness flags
# Useful for filtering to actors with richer data coverage
final_df["has_country_attribution"] = (final_df["country_attribution"] != "unknown").astype(int)
final_df["has_targeted_sector"]     = (final_df["targeted_sectors"]    != "unknown").astype(int)
final_df["has_cve_mapping"]         = (final_df["associated_cve_count"] > 0).astype(int)

print("Scores calculated.")
print(f"  Actors with capability data:     {final_df['has_cve_mapping'].sum():,}")
print(f"  Actors with country attribution: {final_df['has_country_attribution'].sum():,}")
print(f"  Actors with sector data:         {final_df['has_targeted_sector'].sum():,}")


Scores calculated.
  Actors with capability data:     112
  Actors with country attribution: 174
  Actors with sector data:         154


## Cell 20: Export Final Dataset & Preview

Saves `actor_master_dataset_final.csv` to output folder and displays
a summary preview sorted by composite risk score.

This is the primary deliverable of the praxis data engineering pipeline.


In [ ]:
output_file = out_path("actor_master_dataset_final.csv")
final_df.to_csv(output_file, index=False)

print("=" * 60)
print("PRAXIS PIPELINE COMPLETE")
print("=" * 60)
print(f"Final dataset: {output_file}")
print(f"Rows:          {len(final_df):,} actors")
print(f"Columns:       {len(final_df.columns)}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")

# Preview the top actors by composite risk score
preview_cols = [
    "group_id", "actor_name", "actor_type", "country_attribution",
    "targeted_sectors", "technique_count", "associated_cve_count",
    "avg_cvss_score", "kev_count",
    "acled_political_violence_events_total", "geo_instability_score",
    "capability_score", "opportunity_score", "activity_score",
    "composite_risk_score",
]

print("\nTop 20 actors by composite risk score:")
display(
    final_df[preview_cols]
    .sort_values("composite_risk_score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)


PRAXIS PIPELINE COMPLETE
Final dataset: /content/drive/MyDrive/Colab Notebooks/output/actor_master_dataset_final.csv
Rows:          174 actors
Columns:       30

All outputs saved to: /content/drive/MyDrive/Colab Notebooks/output

Top 20 actors by composite risk score:


,group_id,actor_name,actor_type,country_attribution,targeted_sectors,technique_count,associated_cve_count,avg_cvss_score,kev_count,acled_political_violence_events_total,geo_instability_score,capability_score,opportunity_score,activity_score,composite_risk_score
0,G0007,APT28,State,Russia,Chemical; Defense Industrial Base; Nuclear Rea...,93,24.0,8.112500,20.0,39560.0,0.132453,0.771579,0.132453,1.000000,0.712280
1,G1003,Ember Bear,State,Russia; Ukraine,Communications; Government Facilities; Informa...,47,9.0,8.866667,7.0,298672.0,1.000000,0.599876,1.000000,0.360000,0.607938
2,G0047,Gamaredon Group,None,Russia; Ukraine,Defense Industrial Base; Government Facilities...,70,2.0,7.800000,2.0,298672.0,1.000000,0.716866,1.000000,0.093333,0.586433
3,G0094,Kimsuky,State,Japan; North Korea; Russia; South Korea; Unite...,Critical Manufacturing; Education; Energy; Gov...,130,6.0,9.100000,5.0,39560.0,0.132453,0.938908,0.132453,0.250000,0.570944
4,G0059,Magic Hound,None,Iran,Defense Industrial Base; Education; Government...,78,12.0,8.866667,11.0,5342.0,0.017886,0.753829,0.016856,0.530000,0.539286
5,G0096,APT41,State,China,Commercial Facilities; Communications; Educati...,82,13.0,8.625000,9.0,2830.0,0.009475,0.707000,0.009475,0.486667,0.501395
6,G0027,Threat Group-3390,None,China,Critical Manufacturing; Defense Industrial Bas...,57,15.0,8.386667,12.0,2830.0,0.009475,0.631731,0.009475,0.610000,0.500760
7,G1031,Saint Bear,None,Georgia; Russia; Ukraine,Government Facilities; Information Technology,18,2.0,8.800000,2.0,298672.0,1.000000,0.542204,1.000000,0.093333,0.499102
8,G0034,Sandworm Team,State,Georgia; Russia; United Kingdom,Chemical; Defense Industrial Base; Energy; Gov...,79,7.0,9.057143,6.0,39560.0,0.132453,0.747620,0.132453,0.296667,0.489301
9,G0016,APT29,State,Russia; United Kingdom,Government Facilities; NGO; Transportation Sys...,66,8.0,9.325000,7.0,39560.0,0.132453,0.708871,0.132453,0.343333,0.483926


## Cell 21: Debugging Code
The remaining code blocks were used to debug different aspects of the code and provide descriptive information.

In [ ]:
#Code used to check on amount of CVEs in NVD that do not have CVSS scores
#This code was used when I was debugging the NVD code, but also provides insight into how not all CVEs have a CVSS score
missing_cvss_count = nvd_cvss['cvss_score'].isnull().sum()
total_cves = len(nvd_cvss)

print(f"Total CVEs in nvd_cvss: {total_cves:,}")
print(f"CVEs with missing CVSS scores: {missing_cvss_count:,}")
print(f"Percentage of CVEs with missing CVSS scores: {missing_cvss_count / total_cves:.2%}")

Total CVEs in nvd_cvss: 350,896
CVEs with missing CVSS scores: 19,698
Percentage of CVEs with missing CVSS scores: 5.61%


In [ ]:
#This code was used when I was debugging the NVD code cell
print("nvd_cvss in memory:      ", len(nvd_cvss), "rows")
print("nvd_cvss with scores:    ", nvd_cvss["cvss_score"].notna().sum(), "rows")
print()
print("reviewed_actor_cve rows: ", len(reviewed_actor_cve))
print("Sample CVE IDs from reviewed_actor_cve:")
print(reviewed_actor_cve["cve_id"].head(10).tolist())
print()
print("Sample CVE IDs from nvd_cvss:")
print(nvd_cvss["cve_id"].head(10).tolist())
print()
# Check how many CVEs in the actor mapping actually exist in nvd_cvss
matched = reviewed_actor_cve["cve_id"].isin(nvd_cvss["cve_id"]).sum()
total   = len(reviewed_actor_cve)
print(f"CVEs in actor mapping found in nvd_cvss: {matched} / {total}")

nvd_cvss in memory:       350896 rows
nvd_cvss with scores:     331198 rows

reviewed_actor_cve rows:  464
Sample CVE IDs from reviewed_actor_cve:
['CVE-2024-30088', 'CVE-2020-1472', 'CVE-2021-31207', 'CVE-2017-0262', 'CVE-2018-0798', 'CVE-2020-1472', 'CVE-2023-34048', 'CVE-2021-26855', 'CVE-2019-18935', 'CVE-2019-19871']

Sample CVE IDs from nvd_cvss:
['CVE-2009-2466', 'CVE-2010-1233', 'CVE-2019-6535', 'CVE-2010-5305', 'CVE-2018-14668', 'CVE-2018-14669', 'CVE-2018-14670', 'CVE-2018-14671', 'CVE-2018-14672', 'CVE-2019-18657']

CVEs in actor mapping found in nvd_cvss: 461 / 464


## Cell 22: Prepare the data for Mintab
Two  columns need to be added to the actor dataset produced by the code above to be able to do the hypothesis testing in Minitab.

**risk_group**:
Code is added to calculcate the top-25% composite_risk_score threshold for this dataset. The result shown below is 0.3449. Actors at or above this value are labeled High Risk (n=44); all others are Low Risk (n=130). The following values are given for each actor in the risk_group column:
* 1 = High Risk (composite_risk_score ≥ 0.345)
* 0 = Low Risk

 **government or defense targets:**
Hypothesis 4 examines the actors that target government and defense sectors vs other sectors. So, a column needs to be added to label these actors according to their known targeted sector.  The following values are given for each actor in the gov_defense target column:
* 1 = targeted sectors contains "Government Facilities" OR "Defense Industrial Base"
*   0 = targeted sectors doesn't include government or defense

  

In [ ]:
import pandas as pd
import os

df = pd.read_csv(out_path("actor_master_dataset_Mintab.csv"))

# H1 / H2: Risk group (top 25% = High Risk)
threshold = df["composite_risk_score"].quantile(0.75)
df["risk_group"] = (df["composite_risk_score"] >= threshold).astype(int)

# H4: Government or Defense target flag
gov_def_keywords = ["Government Facilities", "Defense Industrial Base"]
df["gov_defense_target"] = df["targeted_sectors"].fillna("").apply(
    lambda x: int(any(k in x for k in gov_def_keywords))
)

df.to_csv(out_path("actor_master_dataset_Mintab_derived.csv"), index=False)
print(f"Threshold: {threshold:.4f}")
print(f"High risk actors: {df['risk_group'].sum()}")
print(f"Gov/Defense actors: {df['gov_defense_target'].sum()}")

Threshold: 0.3449
High risk actors: 44
Gov/Defense actors: 88
